# 10 — Data Engineering for Network LLMs

**Network LLM Engineering — Part III — Adaptation**

### Learning goals
- Design SFT, preference, retrieval and evaluation datasets separately
- Prevent leakage and low-quality synthetic data
- Build metadata and provenance into the dataset

In [ ]:
from pathlib import Path

def find_root():
    for p in [Path.cwd(), Path.cwd().parent, Path("/content/network_llm_engineering_course")]:
        if (p / "data" / "glossary.csv").exists():
            return p
    raise FileNotFoundError("Run from the extracted network_llm_engineering_course folder.")

ROOT = find_root()
DATA = ROOT / "data"
print("Course root:", ROOT)

## Four datasets, four jobs

1. **Knowledge corpus**: RFCs, runbooks, vendor docs -> retrieval/continued pretraining.
2. **SFT data**: prompt -> ideal response -> behavior learning.
3. **Preference data**: prompt -> chosen/rejected -> alignment.
4. **Evaluation data**: held-out tasks with rubrics/expected evidence -> measurement.

Mixing these roles leads to weak experiments.

In [ ]:
import json, pandas as pd
train_path = DATA/"network_sft_train.jsonl"
if train_path.exists():
    rows = [json.loads(x) for x in open(train_path, encoding="utf-8")]
    print("SFT examples:", len(rows))
    display(pd.Series([r["task"] for r in rows]).value_counts())
else:
    print("SFT dataset from course v1 not found; the rest of this lesson still applies.")

## Network-specific quality checklist

A strong incident example should:
- identify observations separately from assumptions,
- avoid fabricated CLI/telemetry,
- use vendor labels when syntax matters,
- prefer checks before config changes,
- include negative/counterexamples,
- cover healthy states as well as failures,
- vary topology and wording,
- preserve source/provenance,
- remove secrets/customer identifiers.

In [ ]:
import re
# Simple leakage check by normalized prompt text.
def norm(s):
    return re.sub(r"\s+"," ", re.sub(r"[^a-z0-9 /.-]","",s.lower())).strip()

if train_path.exists() and (DATA/"network_sft_eval.jsonl").exists():
    train = [json.loads(x) for x in open(train_path, encoding="utf-8")]
    ev = [json.loads(x) for x in open(DATA/"network_sft_eval.jsonl", encoding="utf-8")]
    def user(r): return r["prompt"][-1]["content"]
    overlap = set(norm(user(r)) for r in train) & set(norm(user(r)) for r in ev)
    print("exact normalized overlap:", len(overlap))

### Exercise

Take five real, sanitized incidents and create:
- 3 SFT examples,
- 2 evaluation-only examples.

Do not merely paraphrase the same topology into both sets.